# E10 — Attention / Gating Fusion

**Owner: Bayo**

A gating network learns **per-sample** weights for how much to trust classical vs. deep features.

**Architecture** (`src/fusion/attention_fusion.py`):
```
classical (252) ──→ Linear(252→128) ──┐
                                       ├── concat(256) → gate(2-way softmax) → weighted sum (128) → Linear(128→6)
deep (1280) ─────→ Linear(1280→128) ──┘
```

**Baselines:** E2 macro_f1=0.6476 | E3 macro_f1=0.7848  
**Primary metric:** Macro-F1

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/drive/MyDrive/CV/repo')
    FEAT_DIR  = Path('/content/drive/MyDrive/CV/05_features')
else:
    REPO_ROOT = next(
        (p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / 'src').exists()),
        Path.cwd()
    )
    FEAT_DIR = REPO_ROOT / 'data/processed/features'

MDL_DIR     = REPO_ROOT / 'models'
FIG_DIR     = REPO_ROOT / 'figures/fusion'
RES_DIR     = REPO_ROOT / 'results'
METRICS_CSV = RES_DIR / 'metrics/classification_results.csv'
PRED_DIR    = RES_DIR / 'predictions'

FIG_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT:', REPO_ROOT)
print('FEAT_DIR :', FEAT_DIR, '| exists:', FEAT_DIR.exists())

In [ ]:
import time
import datetime
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score

from src.utils.seed import set_seed, SEED
from src.fusion.attention_fusion import AttentionFusion
from src.evaluation.classification_metrics import (
    compute_all_metrics, save_metrics_row, get_confusion_matrix, CLASSES,
)
from src.evaluation.plots import plot_confusion_matrix

set_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Classes:', CLASSES)

In [ ]:
X_c_train = np.load(FEAT_DIR / 'classical_train_clean_X.npy')  # (45177, 252)
y_train    = np.load(FEAT_DIR / 'classical_train_clean_y.npy')
X_c_val   = np.load(FEAT_DIR / 'classical_val_X.npy')          # (9935, 252)
y_val      = np.load(FEAT_DIR / 'classical_val_y.npy')
X_c_test  = np.load(FEAT_DIR / 'classical_test_X.npy')         # (10553, 252)
y_test     = np.load(FEAT_DIR / 'classical_test_y.npy')

X_d_train = np.load(FEAT_DIR / 'deep_train_X.npy')             # (45177, 1280)
X_d_val   = np.load(FEAT_DIR / 'deep_val_X.npy')
X_d_test  = np.load(FEAT_DIR / 'deep_test_X.npy')

assert X_c_train.shape == (45177, 252)
assert X_d_train.shape == (45177, 1280)
print(f'Train {X_c_train.shape}  Val {X_c_val.shape}  Test {X_c_test.shape}')

In [ ]:
# Normalise each modality separately — fit on train only
scaler_c = StandardScaler().fit(X_c_train)
scaler_d = StandardScaler().fit(X_d_train)

Xc_tr = scaler_c.transform(X_c_train).astype(np.float32)
Xc_va = scaler_c.transform(X_c_val).astype(np.float32)
Xc_te = scaler_c.transform(X_c_test).astype(np.float32)

Xd_tr = scaler_d.transform(X_d_train).astype(np.float32)
Xd_va = scaler_d.transform(X_d_val).astype(np.float32)
Xd_te = scaler_d.transform(X_d_test).astype(np.float32)

# Save scalers for reproducibility
joblib.dump(scaler_c, MDL_DIR / 'e10_scaler_classical.pkl')
joblib.dump(scaler_d, MDL_DIR / 'e10_scaler_deep.pkl')
print('Scalers saved.')

## Build & Train Attention Fusion Model

In [ ]:
set_seed(SEED)

model = AttentionFusion(
    classical_dim=252,
    deep_dim=1280,
    hidden_dim=128,
    num_classes=6,
    dropout=0.3,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal params: {total_params:,}')

In [ ]:
# ── Training with early stopping ─────────────────────────────────────────────
EPOCHS     = 100
BATCH_SIZE = 256
LR         = 1e-3
PATIENCE   = 10     # early stop if val macro-F1 doesn't improve

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6
)
criterion = nn.CrossEntropyLoss()

train_ds = TensorDataset(
    torch.from_numpy(Xc_tr), torch.from_numpy(Xd_tr), torch.LongTensor(y_train)
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

history = {'train_loss': [], 'val_loss': [], 'val_macro_f1': []}
best_val_f1 = -1.0
patience_cnt = 0
best_state   = None

t_train_start = time.time()
for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    total_loss = 0.0
    for xc, xd, yt in train_loader:
        xc, xd, yt = xc.to(DEVICE), xd.to(DEVICE), yt.to(DEVICE)
        logits = model(xc, xd)
        loss   = criterion(logits, yt)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)

    # Validate
    model.eval()
    with torch.no_grad():
        xc_v = torch.from_numpy(Xc_va).to(DEVICE)
        xd_v = torch.from_numpy(Xd_va).to(DEVICE)
        logits_v = model(xc_v, xd_v)
        val_loss = criterion(logits_v, torch.LongTensor(y_val).to(DEVICE)).item()
        val_preds = logits_v.argmax(dim=1).cpu().numpy()
    val_f1 = float(f1_score(y_val, val_preds, average='macro', zero_division=0))

    history['train_loss'].append(avg_loss)
    history['val_loss'].append(val_loss)
    history['val_macro_f1'].append(val_f1)

    scheduler.step(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1

    if epoch % 10 == 0 or patience_cnt == PATIENCE:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:3d}/{EPOCHS}  train_loss={avg_loss:.4f}  '
              f'val_loss={val_loss:.4f}  val_f1={val_f1:.4f}  lr={lr_now:.2e}')

    if patience_cnt >= PATIENCE:
        print(f'Early stopping at epoch {epoch} (no val improvement for {PATIENCE} epochs)')
        break

total_train_s = time.time() - t_train_start
print(f'\nTraining done in {total_train_s:.0f}s  |  best val macro-F1: {best_val_f1:.4f}')

# Restore best weights
model.load_state_dict(best_state)

In [ ]:
model_path = MDL_DIR / 'e10_attention_model.pt'
torch.save({'model_state_dict': model.state_dict(),
            'classical_dim': 252, 'deep_dim': 1280,
            'hidden_dim': 128, 'num_classes': 6}, model_path)
model_size_mb = model_path.stat().st_size / 1e6
print(f'Model saved: {model_path}  ({model_size_mb:.2f} MB)')

# Save training history
with open(RES_DIR / 'e10_training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print('History saved.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='train loss')
ax1.plot(history['val_loss'],   label='val loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('E10 — Training Loss'); ax1.legend()

ax2.plot(history['val_macro_f1'], color='green', label='val macro-F1')
ax2.axhline(0.7848, color='red', ls='--', lw=1, label='E3 baseline 0.7848')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro-F1')
ax2.set_title('E10 — Validation Macro-F1'); ax2.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'E10_training_curves.png', dpi=150)
plt.show()

## Evaluate on Test Set

In [ ]:
model.eval()
with torch.no_grad():
    xc_te_t = torch.from_numpy(Xc_te).to(DEVICE)
    xd_te_t = torch.from_numpy(Xd_te).to(DEVICE)

    t0 = time.time()
    logits_te = model(xc_te_t, xd_te_t)
    clf_time_s_e10 = time.time() - t0

    proba_te = torch.softmax(logits_te, dim=1).cpu().numpy()
    preds_te = np.argmax(proba_te, axis=1)

print(f'E10  test macro-F1 : {f1_score(y_test, preds_te, average="macro", zero_division=0):.4f}')
print(f'E10  test accuracy : {accuracy_score(y_test, preds_te):.4f}')

np.save(PRED_DIR / 'E10_predictions.npy',   preds_te)
np.save(PRED_DIR / 'E10_probabilities.npy', proba_te)

## Attention Weight Analysis

Extract the 2-way softmax gate for each test crop.  
Column 0 = classical weight, Column 1 = deep weight.

In [ ]:
# Attach a forward hook to capture gate outputs
attn_outputs = []

def _hook(module, input, output):
    attn_outputs.append(output.detach().cpu().numpy())

# The gate is the Softmax layer (last layer of model.gate)
hook_handle = model.gate[-1].register_forward_hook(_hook)

model.eval()
with torch.no_grad():
    _ = model(xc_te_t, xd_te_t)

hook_handle.remove()
attn_weights = np.vstack(attn_outputs)   # (N_test, 2)

print(f'Attention weights shape: {attn_weights.shape}')
print(f'Row-sum check (should be ~1.0): {attn_weights[0].sum():.4f}')
print(f'Global avg — classical: {attn_weights[:,0].mean():.4f}  deep: {attn_weights[:,1].mean():.4f}')

np.save(PRED_DIR / 'E10_attention_weights.npy', attn_weights)

In [ ]:
records = []
for cls_id, cls_name in enumerate(CLASSES):
    mask = (y_test == cls_id)
    avg  = attn_weights[mask].mean(axis=0)
    records.append({
        'class'           : cls_name,
        'avg_classical_w' : round(float(avg[0]), 4),
        'avg_deep_w'      : round(float(avg[1]), 4),
        'n_samples'       : int(mask.sum()),
    })

df_attn = pd.DataFrame(records)
df_attn.to_csv(PRED_DIR / 'E10_attention_by_class.csv', index=False)
print(df_attn.to_string(index=False))

In [ ]:
heatmap_data = df_attn[['avg_classical_w', 'avg_deep_w']].values  # (6, 2)

fig, ax = plt.subplots(figsize=(5, 5))
sns.heatmap(
    heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd',
    xticklabels=['Classical', 'Deep'],
    yticklabels=CLASSES,
    vmin=0, vmax=1, ax=ax,
    linewidths=0.5,
)
ax.set_title('E10 — Average Attention Weights per Class')
ax.set_xlabel('Feature branch')
plt.tight_layout()
plt.savefig(FIG_DIR / 'E10_attention_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Validation macro-F1 (best epoch)
val_pred_best  = model(torch.from_numpy(Xc_va).to(DEVICE),
                       torch.from_numpy(Xd_va).to(DEVICE))
val_pred_best  = val_pred_best.argmax(dim=1).detach().cpu().numpy()
val_f1_e10     = float(f1_score(y_val, val_pred_best, average='macro', zero_division=0))

cm_e10 = get_confusion_matrix(y_test, preds_te)
plot_confusion_matrix(cm_e10, 'E10 — Attention Fusion', str(FIG_DIR / 'E10_confusion_matrix.png'))

row_e10 = compute_all_metrics(
    y_true=y_test, y_pred=preds_te, y_prob=proba_te,
    experiment_id='E10', model_name='AttentionFusion_PyTorch',
    feature_dim=128,  # fused hidden dim
    extraction_time_s=0.0, n_train_samples=len(y_train),
    classification_time_s=clf_time_s_e10, n_test_samples=len(y_test),
    augmentation='none',
)
row_e10.update({
    'val_macro_f1' : round(val_f1_e10, 4),
    'model_size_mb': round(model_size_mb, 4),
    'epochs_trained': len(history['train_loss']),
    'timestamp'    : datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
})
save_metrics_row(row_e10, str(METRICS_CSV), 'E10')
print(f"E10 → accuracy={row_e10['accuracy']}  macro_f1={row_e10['macro_f1']}")

---
## Summary — All Bayo Experiments vs Baselines

In [ ]:
df_all = pd.read_csv(METRICS_CSV)
cols   = ['experiment', 'model', 'feature_dim', 'accuracy', 'macro_f1',
          'balanced_accuracy', 'inference_ms_per_crop']
show   = df_all[df_all['experiment'].isin(['E2', 'E3', 'E7', 'E8', 'E9', 'E10'])][cols]
print(show.sort_values('macro_f1', ascending=False).to_string(index=False))